# Hide & Seek

I wanted to see whether self-play could teach agents to use boxes and ramps in a small 2D
environment. I tried several versions of PPO, curricula, layouts, and team sizes. The agents
learned some object interaction, but the full sequence I was aiming for never came together.

This notebook looks at the saved policies. Install `uv sync --locked --extra notebook` and
select the repository's `.venv` as your kernel to run the cells.

## The game

Hiders try to stay out of sight. Seekers try to see at least one hider. During the first 144
steps, the seekers are frozen so the hiders can move boxes and lock them in place. The next
216 steps count toward the score.

The ramp is a simplification: agents within its radius can see over interior walls and boxes.
It doesn't allow wall traversal in these runs. The later climbing experiment is separate.

The policies take partially observed positions and velocities as input. Hiders share one
network and seekers share another, with a separate LSTM state for each agent. During training,
each learning team faces a frozen opponent sampled from earlier snapshots.

## Examples

Blue agents are hiders, red agents are seekers. Boxes are orange; the ramp is green.
These episodes were selected to show particular actions.

**Box placement: tight-room, seed 10**

![Hider placing boxes by the doorway](media/barricade.gif)

**Visibility from the ramp: room, seed 136**

![A seeker gains a sightline near the ramp](media/sightline.gif)

The red line shows a sightline that disappears when elevation is disabled. It says nothing
about climbing across the wall. [Two-hider seed 25](media/ramp-lock.gif) shows a ramp lock.

## Saved evaluations

There are 200 seeds (0–199) for each setup and comparison condition, using deterministic
policy actions and no training spawn assists. Each seed starts with fresh physics and LSTM
state so it can be replayed on its own. These runs used Python 3.13.13, Torch 2.12.0, and
Pymunk 7.2.0 on macOS ARM64.

The table below reads the saved data. All three setups use the save-best pair:

- `room`: one hider, two seekers, equal speed; prefix `hs_lstm`.
- `tight-room`: one hider, two seekers in a smaller room; prefix `hs_roomt`.
- `two-hiders`: two hiders, two seekers in the smaller room, seekers 1.4x faster; prefix `hs_2v2`.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd()
presets = ("room", "tight-room", "two-hiders")
results = {name: json.loads((root / "results" / f"{name}.json").read_text()) for name in presets}

rows = ["| Setup | Doorway box | Ramp sightline | Hider ramp lock |",
        "|---|---:|---:|---:|"]
for name, result in results.items():
    m = result["conditions"]["trained"]["mean"]
    rows.append(f"| {name} | {m['barricade_at_end']:.1%} | "
                f"{m['ramp_sightline_fraction']:.1%} | {m['ramp_locked_at_end']:.1%} |")
display(Markdown("\n".join(rows)))

| Setup | Doorway box | Ramp sightline | Hider ramp lock |
|---|---:|---:|---:|
| room | 0.0% | 12.7% | 0.0% |
| tight-room | 14.5% | 8.2% | 0.0% |
| two-hiders | 26.5% | 10.6% | 3.0% |

The doorway and lock columns are percentages of episodes. Sightline is the mean fraction
of play steps. Each measurement has a specific meaning:

- **Doorway box:** a hider-locked box center ends inside the designated doorway region.
  This does not check whether it forms an impassable barricade.
- **Ramp sightline:** at least one seeker sees a hider with elevation but cannot see that same
  hider when elevation is removed from the query. Another seeker may already have a sightline.
- **Ramp lock:** the hider team owns the ramp lock at the end. A lock may not help the hiders.

The six ramp-lock episodes don't establish a coordinated defense. In seed 14, a box is at
the doorway and the ramp is locked, yet the hiders are visible for every play step.

## Comparisons

The same policies were also evaluated with one active seeker and with hider actions set to
zero. The inactive seeker stays in the scene as a physical body; zero-action hiders can still
be pushed. Hidden time measures how often all hiders are out of sight.

In [2]:
rows = ["| Setup | Hidden: trained | Hidden: zero-action hiders | "
        "Sightline: trained | Sightline: one seeker |",
        "|---|---:|---:|---:|---:|"]
for name, result in results.items():
    trained = result["conditions"]["trained"]["mean"]
    still = result["conditions"]["zero_action_hiders"]["mean"]
    single = result["conditions"]["one_active_seeker"]["mean"]
    rows.append(f"| {name} | {trained['hidden_fraction']:.1%} | "
                f"{still['hidden_fraction']:.1%} | {trained['ramp_sightline_fraction']:.1%} | "
                f"{single['ramp_sightline_fraction']:.1%} |")
display(Markdown("\n".join(rows)))

| Setup | Hidden: trained | Hidden: zero-action hiders | Sightline: trained | Sightline: one seeker |
|---|---:|---:|---:|---:|
| room | 83.7% | 78.0% | 12.7% | 10.4% |
| tight-room | 75.2% | 71.7% | 8.2% | 4.1% |
| two-hiders | 66.3% | 58.5% | 10.6% | 6.3% |

## What changed in the interpretation

An earlier metric counted being near the ramp as tool use. The current check asks whether
that proximity changes visibility. The old description of agents transporting ramps and
climbing walls went beyond what the evaluation showed.

The ramp does move in some episodes. Mean final displacement in the two-hider run is 44.62
pixels in a 600-pixel arena; it is 58.25 pixels with zero-action hiders. That alone doesn't
show purposeful transport to a useful location.

I also changed too many things between runs to conclude that geometry, rather than compute,
was responsible for the results. The curricula, team sizes, speeds, and checkpoint histories
all differ. There was no set of independent training seeds or test on unseen layouts.

Some numbers differ from the original journal. The old evaluator reused environments and
used a looser ramp metric. I haven't traced every difference to its cause. The saved JSON
files describe the current evaluation, including versions, settings, and checkpoint hashes.

## Run a short evaluation

This checks that the saved policies load and play. Two episodes aren't enough to estimate
the frequency of a rare behavior. Use `--episodes 200` to repeat the full evaluation.

In [3]:
import subprocess
import sys

run = subprocess.run(
    [sys.executable, "evaluate.py", "--preset", "two-hiders", "--episodes", "2"],
    cwd=root, check=True, capture_output=True, text=True,
)
print(run.stdout)


trained (2 episodes)
  hidden_fraction: 94.68%
  barricade_at_end: 0.00%
  near_ramp_fraction: 1.62%
  ramp_sightline_fraction: 1.62%
  ramp_locked_at_end: 0.00%
  ramp_net_displacement_px: 0.00 px
  ramp_max_displacement_px: 0.00 px

one_active_seeker (2 episodes)
  hidden_fraction: 100.00%
  barricade_at_end: 50.00%
  near_ramp_fraction: 0.00%
  ramp_sightline_fraction: 0.00%
  ramp_locked_at_end: 0.00%
  ramp_net_displacement_px: 0.00 px
  ramp_max_displacement_px: 0.00 px

zero_action_hiders (2 episodes)
  hidden_fraction: 81.71%
  barricade_at_end: 0.00%
  near_ramp_fraction: 10.65%
  ramp_sightline_fraction: 10.65%
  ramp_locked_at_end: 0.00%
  ramp_net_displacement_px: 11.22 px
  ramp_max_displacement_px: 11.22 px



To replay an example:

```bash
uv run python demo.py --preset room --seed 136 --output media/replay.gif
```

[NOTES.md](NOTES.md) has the training history, including the unstable runs, save-best issue,
and reverse-curriculum attempts. The [experiment index](experiments/README.md) lists their scripts.